# B — Triples to graph (CPU)

Everything downstream of extraction. Reads notebook A's `biology_all_triples.csv`, builds
a node/edge graph, and exports it for Neo4j and the verifier. No GPU — iterate freely.

**Two settings at the top are research decisions, not defaults to accept blindly:**
`DROP_RELATIONS` (which relation types are too unreliable to keep) and `MENTION_EDGE_TYPE`
(whether entity mention-links inherit the original relation or get a neutral one). Both
are explained where they appear.

In [ ]:
import re, glob, collections
from pathlib import Path
import pandas as pd

# ---- decisions ----------------------------------------------------------
# Relations the quality review shows are unreliable. Empty until that review exists —
# do not guess. 'অবস্থান' is the current suspect (see quality_sample.csv).
DROP_RELATIONS = []

# How to type an edge recovered by finding entity B mentioned inside the object text of
# a triple about entity A. Inheriting the original relation asserts things the textbook
# did not say — "মূল --[অংশ]--> খাদ্য" from "খাদ্য জমা থাকে" claims food is PART OF a root.
# A neutral type keeps the connectivity without the false claim. None = inherit.
MENTION_EDGE_TYPE = "সম্পর্কিত"
# -------------------------------------------------------------------------

hits = (glob.glob("/kaggle/input/**/biology_all_triples.csv", recursive=True)
        or glob.glob("kg/triples/biology_all_triples.csv")
        or glob.glob("../kg/triples/biology_all_triples.csv"))
if not hits:
    raise SystemExit("biology_all_triples.csv not found — attach notebook A's output.")

t = pd.read_csv(hits[0])
t["subject"] = t.subject.astype(str).str.strip()
t["object"] = t.object.astype(str).str.strip()
print(f"{len(t)} triples, {t.chapter_no.nunique()} chapters, {t.relation.nunique()} relations")

if DROP_RELATIONS:
    before = len(t)
    t = t[~t.relation.isin(DROP_RELATIONS)].reset_index(drop=True)
    print(f"dropped {before - len(t)} triples from {DROP_RELATIONS}")

## 1 — Canonical surface forms

Only whitespace and punctuation are normalised. Aggressive Bangla suffix stripping was
measured and rejected: it merged 14 of 1,121 subjects while risking real distinctions,
so inflection is not what makes this graph sparse.

In [ ]:
WORD = re.compile(r"[ঀ-৿]+|[A-Za-z0-9]+")


def canon(s):
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s.strip(" ।,:;()[]\"'-–—")


t["subject"] = t.subject.map(canon)
t["object"] = t.object.map(canon)
t = t[(t.subject.str.len() > 1) & (t.object.str.len() > 1)].reset_index(drop=True)

# A subject should be a noun phrase. A long one is usually a clause the extractor
# mis-assigned — "ইচ্ছানুযায়ী সংকুচিত বা প্রসারিত হয়" appeared as a subject. Flagged,
# not dropped: the review decides whether the cutoff is right.
t["subj_tokens"] = t.subject.map(lambda s: len(WORD.findall(s)))
t["subj_suspect"] = t.subj_tokens > 5
print(f"subjects longer than 5 tokens (likely clauses): {t.subj_suspect.sum()} "
      f"({t.subj_suspect.mean()*100:.1f}%)")
t[t.subj_suspect].subject.head(5).to_list()

## 2 — Nodes

An entity is any canonical subject. Objects are kept as literal values on the fact edge
rather than promoted to nodes — most are descriptive phrases, not entities.

In [ ]:
ent = sorted(set(t.subject))
nodes = pd.DataFrame({"node_id": range(len(ent)), "label": ent})
nid = dict(zip(nodes.label, nodes.node_id))

chap = t.groupby("subject").chapter_no.agg(lambda s: sorted(set(s)))
deg = t.subject.value_counts()
nodes["chapters"] = nodes.label.map(lambda l: ",".join(map(str, chap.get(l, []))))
nodes["n_facts"] = nodes.label.map(lambda l: int(deg.get(l, 0)))

print(f"{len(nodes)} entity nodes")
print(f"appearing in >1 chapter: {(nodes.chapters.str.contains(',')).sum()}")
nodes.sort_values('n_facts', ascending=False).head(8)

## 3 — Fact edges

One per triple: the literal claim the textbook makes. These are what the verifier checks
a tutor's answer against.

In [ ]:
facts = pd.DataFrame({
    "triple_id": t.triple_id,
    "head_id": t.subject.map(nid),
    "head": t.subject,
    "relation": t.relation,
    "value": t.object,
    "chapter_no": t.chapter_no,
    "chunk_id": t.chunk_id,
    "subj_suspect": t.subj_suspect,
})
print(f"{len(facts)} fact edges")
print(facts.relation.value_counts().to_string())

## 4 — Mention edges (entity ↔ entity)

Whole-token matching, so `তন্দ্র` cannot match inside `টিস্যুতন্দ্র`. This is what gives the
graph paths to traverse; without it, 93% of facts are dead ends.

In [ ]:
SUFFIXES = ["গুলোর", "গুলোকে", "গুলো", "টিকে", "গুলি", "দের", "টির", "টি",
            "য়ের", "এর", "কে", "ের", "রা", "র"]
GENERIC = {"মাধ্যম", "ধরন", "জিনিস", "সময়", "স্থান", "গুরুত্ব", "নাম",
           "ব্যাপার", "কারণ", "ফলে", "দিক"}


def lemma(w):
    for s in SUFFIXES:
        if w.endswith(s) and len(w) - len(s) >= 3:
            return w[: -len(s)]
    return w


def toks(s):
    return [lemma(w) for w in WORD.findall(s)]


vocab = {e: tuple(toks(e)) for e in ent}
vocab = {e: tk for e, tk in vocab.items()
         if tk and len("".join(tk)) >= 4 and e not in GENERIC}
by_len = collections.defaultdict(dict)
for e_, tk in vocab.items():
    by_len[len(tk)][tk] = e_
max_n = max(by_len)


def mentions(text):
    tk, found, covered = toks(text), [], set()
    for n in range(max_n, 0, -1):
        table = by_len.get(n)
        if not table:
            continue
        for i in range(len(tk) - n + 1):
            if any(j in covered for j in range(i, i + n)):
                continue
            hit = table.get(tuple(tk[i:i + n]))
            if hit:
                found.append(hit)
                covered.update(range(i, i + n))
    return found


rows = []
for r in t.itertuples():
    for tail in mentions(r.object):
        if tail == r.subject:
            continue
        rows.append({
            "triple_id": r.triple_id,
            "head_id": nid[r.subject], "head": r.subject,
            "tail_id": nid[tail], "tail": tail,
            "relation": MENTION_EDGE_TYPE or r.relation,
            "source_relation": r.relation,
            "chapter_no": r.chapter_no,
        })

ment = pd.DataFrame(rows).drop_duplicates(subset=["head", "tail", "relation"])
linked_triples = ment.triple_id.nunique()
print(f"{len(ment)} mention edges over {linked_triples} triples "
      f"({linked_triples/len(t)*100:.0f}% of facts now connect to another entity)")
print(f"edge type: {MENTION_EDGE_TYPE or 'inherited from source relation'}")

## 5 — Is it actually a graph?

The proposal's verification method relies on path-consistency, which needs paths. A graph
that is mostly isolated nodes cannot support it, so this is the number that decides whether
the method as written is viable.

In [ ]:
adj = collections.defaultdict(set)
for r in ment.itertuples():
    adj[r.head_id].add(r.tail_id)
    adj[r.tail_id].add(r.head_id)

seen, comps = set(), []
for n in nodes.node_id:
    if n in seen:
        continue
    stack, comp = [n], []
    seen.add(n)
    while stack:
        c = stack.pop()
        comp.append(c)
        for nb in adj[c]:
            if nb not in seen:
                seen.add(nb)
                stack.append(nb)
    comps.append(comp)

comps.sort(key=len, reverse=True)
isolated = sum(1 for c in comps if len(c) == 1)
print(f"nodes: {len(nodes)}   mention edges: {len(ment)}")
print(f"connected components: {len(comps)}")
print(f"largest component: {len(comps[0])} nodes ({len(comps[0])/len(nodes)*100:.0f}%)")
print(f"isolated nodes: {isolated} ({isolated/len(nodes)*100:.0f}%)")
print(f"component sizes (top 8): {[len(c) for c in comps[:8]]}")

## 6 — Export

In [ ]:
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("kg/graph")
OUT.mkdir(parents=True, exist_ok=True)

nodes.to_csv(OUT / "nodes.csv", index=False)
facts.to_csv(OUT / "fact_edges.csv", index=False)
ment.to_csv(OUT / "mention_edges.csv", index=False)
print(f"wrote {len(nodes)} nodes, {len(facts)} fact edges, {len(ment)} mention edges -> {OUT}")

## 7 — Figures for the paper

**Bangla in matplotlib needs care.** The default Agg backend maps glyphs in logical order
and never applies Indic reordering, so vowel signs land in the wrong place — `কিডনির রোগ`
silently renders as `কডিনরি রোগ`. Any Bangla reader spots it immediately; matplotlib gives
no warning.

Two independent fixes are applied below:

1. **Raster:** mplcairo with libraqm shapes text through HarfBuzz. Available on Linux
   (so it works on Kaggle); the Windows wheel ships without libraqm, so PNGs generated
   locally on Windows will be wrong.
2. **Vector:** `svg.fonttype = "none"` writes real `<text>` elements rather than baking in
   outlines, so shaping is left to whatever renders the SVG. This works everywhere and is
   the format to put in the paper.

The check at the end of the setup cell reports which path is active. If it says raqm is
off, trust the SVGs and not the PNGs.

In [ ]:
!apt-get -qq install -y libraqm0 fonts-noto-core > /dev/null 2>&1 || true
!pip install -q mplcairo fonttools networkx 2>/dev/null

import os, urllib.request
import matplotlib

SHAPED = False
try:
    import mplcairo
    mplcairo.set_options(raqm=True)
    matplotlib.use("module://mplcairo.base")
    SHAPED = mplcairo.get_options().get("raqm", False)
except Exception as e:
    matplotlib.use("Agg")
    print("mplcairo/raqm unavailable:", type(e).__name__)

import matplotlib.pyplot as plt
from matplotlib import font_manager
import networkx as nx

# Must live under /kaggle/working or Kaggle discards it: OUT.parent there is
# /kaggle, which is outside the captured output directory.
FIGS = (Path("/kaggle/working/figures") if Path("/kaggle/working").exists()
        else Path("kg/figures"))
FIGS.mkdir(parents=True, exist_ok=True)


def bengali_font():
    """Verify the font actually covers the Bengali block — a font named for Bangla that
    lacks the glyphs renders empty boxes with no error."""
    from fontTools.ttLib import TTFont
    probe = [ord(c) for c in "অকরক্তজীব"]
    for f in font_manager.fontManager.ttflist:
        try:
            tt = TTFont(f.fname, fontNumber=0, lazy=True)
            cm = tt.getBestCmap() or {}
            tt.close()
            if all(cp in cm for cp in probe):
                return f.name
        except Exception:
            continue
    url = ("https://github.com/google/fonts/raw/main/ofl/notosansbengali/"
           "NotoSansBengali%5Bwdth%2Cwght%5D.ttf")
    path = FIGS / "NotoSansBengali.ttf"
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    font_manager.fontManager.addfont(str(path))
    return font_manager.FontProperties(fname=str(path)).get_name()


FONT = bengali_font()
plt.rcParams["font.family"] = FONT
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["axes.unicode_minus"] = False

INK, GRID = "#23252b", "#d8d6d2"
PAL = ["#3f7d6e", "#7a9e57", "#c6893f", "#a8553f", "#6b6f8a"]


def save(fig, name):
    for ext in ("png", "svg"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=200, facecolor="white", bbox_inches="tight")
    plt.close(fig)


print(f"font: {FONT}")
print(f"harfbuzz shaping: {SHAPED}"
      + ("" if SHAPED else "  <-- PNGs will have mis-ordered Bangla; use the SVGs"))

### Figure 1 — what the graph contains, and how good it is

Pairs the size of each relation against its manually measured precision, so the reader can see that the largest relations are also the most reliable.

In [ ]:
rev_hits = (glob.glob("/kaggle/input/**/review_200_reviewed.csv", recursive=True)
            + glob.glob("kg/triples/review_200_reviewed.csv")
            + glob.glob("../kg/triples/review_200_reviewed.csv"))
rev_path = Path(rev_hits[0]) if rev_hits else None

if rev_path is None:
    print("no review file — skipping the precision panel")
else:
    rev = pd.read_csv(rev_path)
    g = (rev.groupby("relation").verdict.value_counts().unstack(fill_value=0)
         .reindex(columns=["correct", "wrong_relation", "wrong"], fill_value=0))
    g["n"] = g.sum(axis=1)
    g["strict"] = g.correct / g.n * 100
    g["fact"] = (g.correct + g.wrong_relation) / g.n * 100
    g = g.join(t.relation.value_counts().rename("total")).sort_values("total", ascending=False)

    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1, 1.15]})
    ax[0].barh(g.index[::-1], g.total[::-1], color=PAL[0])
    ax[0].set_xlabel("triples in the knowledge graph")
    ax[0].set_title("Relation distribution", loc="left", fontsize=11)
    ax[0].set_xlim(0, g.total.max() * 1.16)
    for i, v in enumerate(g.total[::-1]):
        ax[0].text(v + g.total.max() * .015, i, str(v), va="center", fontsize=8.5, color=INK)

    y = list(range(len(g)))
    ax[1].barh([i + .2 for i in y][::-1], g.fact[::-1], height=.38,
               color=PAL[1], label="fact correct")
    ax[1].barh([i - .2 for i in y][::-1], g.strict[::-1], height=.38,
               color=PAL[3], label="relation also correct")
    ax[1].set_yticks(y[::-1]); ax[1].set_yticklabels(g.index[::-1])
    ax[1].set_xlim(0, 118)
    ax[1].set_xlabel(f"precision (%), n={len(rev)} manually labelled")
    ax[1].set_title("Measured precision by relation", loc="left", fontsize=11)
    ax[1].legend(frameon=False, fontsize=9, loc="upper right", bbox_to_anchor=(1.0, 1.02))
    for a in ax:
        a.spines[["top", "right"]].set_visible(False)
        a.tick_params(labelsize=9)
    fig.tight_layout()
    save(fig, "fig1_relations_precision")
    print("fig1 written")

### Figure 2 — a readable slice of the graph

The whole graph is 1,121 nodes and unreadable at page size. An ego network around one hub concept shows the real structure at a scale a reader can follow.

In [ ]:
HUB = "রক্ত"

G = nx.Graph()
for r in ment.itertuples():
    G.add_edge(r.head, r.tail)

if HUB not in G:
    print(f"{HUB} not in the graph — pick another hub")
else:
    nodes = {HUB} | set(G.neighbors(HUB))
    for nb in list(G.neighbors(HUB))[:12]:
        nodes |= set(list(G.neighbors(nb))[:2])
    H = G.subgraph(nodes)

    fig, ax = plt.subplots(figsize=(10, 7.5))
    pos = nx.spring_layout(H, seed=7, k=.85, iterations=200)
    deg = dict(H.degree())
    nx.draw_networkx_edges(H, pos, ax=ax, edge_color=GRID, width=1.1)
    nx.draw_networkx_nodes(H, pos, ax=ax,
                           node_size=[130 + 80 * deg[n] for n in H],
                           node_color=[PAL[0] if n == HUB else PAL[4] for n in H],
                           linewidths=0)
    for n, (x, y_) in pos.items():
        ax.text(x, y_ - .055, n, fontsize=8, ha="center", va="top",
                color=INK, fontfamily=FONT)
    ax.set_title(f"Curriculum subgraph around \u201c{HUB}\u201d "
                 f"({H.number_of_nodes()} concepts, {H.number_of_edges()} links)",
                 loc="left", fontsize=11)
    ax.margins(.12)
    ax.axis("off")
    fig.tight_layout()
    save(fig, "fig2_subgraph")
    print(f"fig2 written — {H.number_of_nodes()} nodes, {H.number_of_edges()} edges")

### Figure 3 — coverage across the curriculum

In [ ]:
ti_hits = (glob.glob("/kaggle/input/**/biology_9_10_chapter_titles.csv", recursive=True)
           + glob.glob("data/biology_9_10_chapter_titles.csv")
           + glob.glob("../data/biology_9_10_chapter_titles.csv"))
titles_path = Path(ti_hits[0]) if ti_hits else None
titles = (pd.read_csv(titles_path).set_index("chapter_no").title_bn
          if titles_path is not None else pd.Series(dtype=str))

per = t.groupby("chapter_no").size()
fig, ax = plt.subplots(figsize=(9.5, 4.2))
ax.bar(per.index, per.values, color=PAL[0])
ax.set_xticks(per.index)
ax.set_xticklabels([f"{i}\n{str(titles.get(i, ''))[:14]}" for i in per.index],
                   fontsize=7.5, linespacing=1.6)
ax.set_ylabel("triples extracted")
ax.set_title("Coverage across the Class 9\u201310 Biology curriculum", loc="left", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.set_ylim(0, per.max() * 1.12)
for x, v in zip(per.index, per.values):
    ax.text(x, v + per.max() * .02, str(v), ha="center", fontsize=8, color=INK)
fig.tight_layout()
save(fig, "fig3_chapter_coverage")

print("fig3 written")
print("\nfigures:", sorted(p.name for p in FIGS.glob("fig*")))